In [ ]:
%matplotlib inline
import seaborn
import numpy, scipy, matplotlib.pyplot as plt, IPython.display as ipd, sklearn
import librosa, librosa.display
import numpy as np

# Machine Learning

## Supervised Learning

Supervised learning is a machine learning approach where algorithms learn from labeled datasets to make predictions or decisions, by identifying patterns and relationships between input features and corresponding output labels.

We are going to start with MNIST handwritten digits. It's the 'Hello World' of Machine Learning

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import seaborn as sns

The MNIST dataset is a classic dataset in machine learning and computer vision used for handwritten digit classification. It contains:
* 70,000 grayscale images of digits (0–9)
  * 60,000 for training
  * 10,000 for testing

* Each image is:
  * Size: 28×28 pixels
  * Flattened into a vector of 784 pixel values


In [ ]:
# STEP 2: Load MNIST data
train_df = pd.read_csv('../data/mnist_data/mnist_train_small.csv')
test_df = pd.read_csv('../data/mnist_data/mnist_test.csv')

train_df.rename(columns={train_df.columns[0]: 'label'}, inplace=True)
test_df.rename(columns={test_df.columns[0]: 'label'}, inplace=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

* The first column is the label (the digit: 0–9)
* The next 784 columns are flattened pixel values from a 28×28 image

In [ ]:
# STEP 3: Split features and labels
X_train = train_df.drop('label', axis=1)
y_train = train_df['label']

X_test = test_df.drop('label', axis=1)
y_test = test_df['label']

In [ ]:
pixels = X_train.iloc[0].values.reshape(28, 28)
plt.imshow(pixels, cmap='gray')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get and reshape image
pixels = X_train.iloc[0].values.reshape(28, 28)

# Set figure size
plt.figure(figsize=(10, 10))

# Display image
plt.imshow(pixels, cmap='gray', interpolation='nearest')
plt.title(f"Digit: {y_train.iloc[0]}")
plt.axis('off')

# Add text for pixel values
for i in range(28):
    for j in range(28):
        val = int(pixels[i, j])

        # if val > 0:
        plt.text(j, i, str(val),
                 ha='center', va='center',
                 color='red', fontsize=6)

plt.show()


In [ ]:
# STEP 4: Visualize some digits
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    img = X_train.iloc[i].values.reshape(28, 28)
    plt.imshow(img, cmap='gray')
    plt.title(f"Label: {y_train.iloc[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

# Nearest Neighbor

In [ ]:
class NearestNeighbor:
    def __init__(self):
        pass

    def train(self, X, y):
        """ X is N x D where each row is an example. y is 1-dimension of size N """
        # the nearest neighbor classifier simply remembers all the training data
        self.Xtr = X
        self.ytr = y

    def predict(self, X):
        """ X is N x D where each row is an example we wish to predict label for """
        num_test = X.shape[0]

        # make sure output type matches the input type
        Ypred = np.zeros(num_test, dtype = self.ytr.dtype)

        # loop over all test rows
        for i in range(num_test):
            # find the nearest training image to the i'th test image
            # using the L1 distance (sum of absolute value differences)
            distances = np.sum(np.abs(self.Xtr - X[i,:]), axis = 1)
            min_index = np.argmin(distances) # get the index with smallest distance
            Ypred[i] = self.ytr[min_index] # predict the label of the nearest example
        return Ypred
            

In [ ]:
Xtr = X_train.values
ytr = y_train.values

Xte = X_test.values
yte = y_test.values

nn = NearestNeighbor()
nn.train(Xtr, ytr)

In [ ]:
num_test = 100
Xte_subset = Xte[:num_test]

y_pred = nn.predict(Xte_subset)

In [ ]:
y_pred.shape

In [ ]:
accuracy = np.mean(y_pred == yte[:num_test])
print("Accuracy:", accuracy)

In [ ]:
def get_top_k_neighbors(nn, test_image, k=10):
    # compute distance
    distances = np.sum(np.abs(nn.Xtr - test_image), axis=1)

    # get indices of smallest distances
    nearest_indices = np.argsort(distances)[:k]
    nearest_distance = distances[nearest_indices]

    return nearest_indices, nearest_distance

In [ ]:
def visualize_neighbors(nn, Xte, yte, index, k=10):
    test_img = Xte[index]
    test_label = yte[index]

    neighbors, neighbor_distances = get_top_k_neighbors(nn, test_img, k)

    plt.figure(figsize=(12, 2))

    # show test image
    plt.subplot(1, k+1, 1)
    plt.imshow(test_img.reshape(28, 28), cmap='gray')
    plt.title(f"Test\nLabel: {test_label}")
    plt.axis('off')

    # show neighbors
    for i, (idx, dist) in enumerate(zip(neighbors, neighbor_distances)):
        plt.subplot(1, k+1, i+2)
        plt.imshow(nn.Xtr[idx].reshape(28, 28), cmap='gray')
        plt.title(f"{nn.ytr[idx]}\n{dist:.1f}")
        plt.axis('off')

    plt.show()

In [ ]:
visualize_neighbors(nn, Xte, yte, index=4, k=10)

# K-Nearest Neighbor(KNN) Algorithm

K-Nearest Neighbors (KNN) is a simple way to classify things by looking at what’s nearby. Imagine a streaming service wants to predict if a new user is likely to cancel their subscription based on their age. They checks the ages of its existing users and whether they canceled or stayed. If most of the “K” closest users in age of new user canceled their subscription KNN will predict the new user might cancel too. The key idea is that users with similar ages tend to have similar behaviors and KNN uses this closeness to make decisions.

https://www.geeksforgeeks.org/k-nearest-neighbours/

In [ ]:
# STEP 5: Train a k-NN classifier
from sklearn.preprocessing import StandardScaler

# Replace all column names with integer indices (just for consistency)
X_train.columns = range(X_train.shape[1])
X_test.columns = range(X_test.shape[1])


# Normalize the pixel values
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit k-NN
knn = KNeighborsClassifier(n_neighbors=4, weights="distance")
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)


In [ ]:
# STEP 6: Evaluate the model
print(classification_report(y_test, y_pred))

# Confusion matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("MNIST Confusion Matrix")
plt.show()


In [ ]:
# Pick a test index to visualize
index = 800 # <- change this number to try different digits!

# Get the image and label
image = X_test.iloc[index].values.reshape(28, 28)
true_label = y_test.iloc[index]

# Plot the digit
plt.imshow(image, cmap='gray')
plt.title(f"True Label: {true_label}")
plt.axis('off')
plt.show()


In [ ]:
# Scale the test sample (reshape to 2D for model input)
sample = scaler.transform([X_test.iloc[index].values])

# Predict
predicted_label = knn.predict(sample)[0]
print(f"🔍 Model Prediction: {predicted_label}")


In [ ]:
import random

for _ in range(5):
    idx = random.randint(0, len(X_test) - 1)
    img = X_test.iloc[idx].values.reshape(28, 28)
    label = y_test.iloc[idx]
    pred = knn.predict(scaler.transform([X_test.iloc[idx].values]))[0]

    plt.imshow(img, cmap='gray')
    plt.title(f"True: {label} | Predicted: {pred}")
    plt.axis('off')
    plt.show()
